# 🐱 BÀI TẬP 2: CATBOOST VÀ NATIVE CATEGORICAL HANDLING
Notebook này hướng dẫn sử dụng CatBoost - thuật toán Gradient Boosting hàng đầu cho dữ liệu bảng chứa các thuộc tính phân loại (Categorical Features):
1. **Native Categorical Features**: Truyền trực tiếp danh sách cột chữ/phân loại vào CatBoost không cần One-Hot Encoding.
2. **XGBoost vs LightGBM vs CatBoost**: So sánh hiệu năng 3 mô hình trên cùng tập K-Fold Cross Validation.
3. **Weighted Blending**: Tổ hợp kết quả Out-Of-Fold từ cả 3 mô hình.

In [4]:
import numpy as np
import pandas as pd
import random
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# 1. Cố định Seed
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(2026)
print("--- Seed 2026 initialized successfully! ---")

--- Seed 2026 initialized successfully! ---


In [5]:
print("--- BÀI TẬP 1: GIẢ LẬP DỮ LIỆU CÓ THUỘC TÍNH PHÂN LOẠI ---")

n_samples = 600
data = {
    'age': np.random.randint(18, 65, size=n_samples),
    'income': np.random.randn(n_samples) * 50000 + 100000,
    'city': np.random.choice(['HaNoi', 'HCM', 'DaNang', 'CanTho'], size=n_samples),
    'job': np.random.choice(['Engineer', 'Teacher', 'Doctor', 'Artist', 'Student'], size=n_samples),
    'target': np.random.randint(0, 2, size=n_samples)
}
df = pd.DataFrame(data)

# Xử lý kiểu dữ liệu cột categorical cho CatBoost
cat_cols = ['city', 'job']
for col in cat_cols:
    df[col] = df[col].astype(str)

X = df.drop(columns=['target'])
y = df['target']

print("5 dòng dữ liệu đầu tiên:")
print(df.head())

--- BÀI TẬP 1: GIẢ LẬP DỮ LIỆU CÓ THUỘC TÍNH PHÂN LOẠI ---
5 dòng dữ liệu đầu tiên:
   age         income    city       job  target
0   19  198083.130327  CanTho    Doctor       1
1   24  109091.176365  CanTho    Doctor       0
2   44  102528.360024  CanTho   Teacher       0
3   31   84648.231570     HCM   Student       0
4   31  146076.353174   HaNoi  Engineer       1


In [6]:
print("--- BÀI TẬP 2: HUẤN LUYỆN CATBOOST VỚI NATIVE CAT_FEATURES ---")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
oof_catboost = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # CatBoost nhận trực tiếp cat_features là danh sách tên cột
    model_cb = CatBoostClassifier(
        iterations=100,
        learning_rate=0.05,
        depth=6,
        cat_features=cat_cols,
        random_seed=2026,
        verbose=0
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=20)
    
    probs = np.array(model_cb.predict_proba(X_va))
    oof_catboost[val_idx] = probs[:, 1]

acc_cb = accuracy_score(y, (oof_catboost >= 0.5).astype(int))
f1_cb = f1_score(y, (oof_catboost >= 0.5).astype(int))
print(f"CatBoost OOF Accuracy: {acc_cb:.4f} | F1-Score: {f1_cb:.4f}")

--- BÀI TẬP 2: HUẤN LUYỆN CATBOOST VỚI NATIVE CAT_FEATURES ---
CatBoost OOF Accuracy: 0.5267 | F1-Score: 0.6553


In [7]:
print("--- BÀI TẬP 3: SO SÁNH 3 MÔ HÌNH (XGBOOST, LIGHTGBM VÀ CATBOOST) ---")

# Chuyển đổi One-Hot Encoding cho XGBoost và LightGBM
X_encoded = pd.get_dummies(X, columns=cat_cols)

oof_xgb = np.zeros(len(df))
oof_lgb = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_encoded, y)):
    X_tr, y_tr = X_encoded.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X_encoded.iloc[val_idx], y.iloc[val_idx]
    
    # XGBoost
    model_xgb = xgb.XGBClassifier(n_estimators=100, random_state=2026, eval_metric='logloss')
    model_xgb.fit(X_tr, y_tr)
    probs_x = np.array(model_xgb.predict_proba(X_va))
    oof_xgb[val_idx] = probs_x[:, 1]
    
    # LightGBM
    model_lgb = lgb.LGBMClassifier(n_estimators=100, random_state=2026, verbose=-1)
    model_lgb.fit(X_tr, y_tr)
    probs_l = np.array(model_lgb.predict_proba(X_va))
    oof_lgb[val_idx] = probs_l[:, 1]

acc_xgb = accuracy_score(y, (oof_xgb >= 0.5).astype(int))
acc_lgb = accuracy_score(y, (oof_lgb >= 0.5).astype(int))

# Trọng số Blending: 40% CatBoost + 30% XGBoost + 30% LightGBM
oof_blend = 0.4 * oof_catboost + 0.3 * oof_xgb + 0.3 * oof_lgb
acc_blend = accuracy_score(y, (oof_blend >= 0.5).astype(int))

print(f"XGBoost  OOF Accuracy: {acc_xgb:.4f}")
print(f"LightGBM OOF Accuracy: {acc_lgb:.4f}")
print(f"CatBoost OOF Accuracy: {acc_cb:.4f}")
print(f"Weighted Ensemble Blend OOF Accuracy: {acc_blend:.4f}")

--- BÀI TẬP 3: SO SÁNH 3 MÔ HÌNH (XGBOOST, LIGHTGBM VÀ CATBOOST) ---
XGBoost  OOF Accuracy: 0.4950
LightGBM OOF Accuracy: 0.5167
CatBoost OOF Accuracy: 0.5267
Weighted Ensemble Blend OOF Accuracy: 0.5067
